🧩 Bloque 1 — Importación de librerías

Carga todas las librerías necesarias para la automatización: 
- Lectura de archivos Excel (pandas) 
- Conexión con bases de datos SQL Server (sqlalchemy) 
- Manejo de rutas y archivos (pathlib, os, shutil) 
- Operaciones con fechas (datetime)
- Apoyo en cálculos o formatos (numpy, re, unicodedata).

In [1]:
import re
import time
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

🗄️ Bloque 2 — Conexión con la base de datos

Define los parámetros de conexión al servidor SQL Server (server, database, schema, table) 

Construye la cadena de conexión usando SQLAlchemy. 

Esta configuración permite insertar los datos procesados directamente en la base de datos.

In [2]:
server = 'SGF1034'                 
database = 'Habitat'      
schema = 'dbo'
table = 'VOLVEK_ACUMULADO_STOCK'
 
connection_string = (
    f"mssql+pyodbc://{server}/{database}"
    f"?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server"
)
 
engine = create_engine(
    connection_string,
    fast_executemany=True,
    pool_pre_ping=True,
    future=True,
)

📂 Bloque 3 — Ruta y configuración del archivo origen

Establece la carpeta donde se buscará el archivo Excel más reciente y define las extensiones válidas. 

Además, indica el nombre de la hoja preferida a leer dentro del archivo, usando una lista de alternativas por si el nombre varía.

In [3]:
RUTA_ARCHIVOS = Path(r"C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK STOCK")
EXTS = (".xlsx",)  # Agregar más extensiones si es necesario

PREFERRED_SHEETS = ["Base Stock", "Base cesantia SURA Stock", "Base"]
FALLBACK_SHEET_INDEX = 0

⚙️ Bloque 4 — Funciones de normalización

Contiene funciones auxiliares para estandarizar textos y nombres de columnas:
- Elimina tildes y caracteres especiales (_strip_accents).
- Convierte encabezados a formato estándar (normalize_name).
- Incluye funciones opcionales para limpiar y transformar datos numéricos y fechas, que pueden usarse si el origen varía.

In [4]:
def _strip_accents(s: str) -> str:
    import unicodedata
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s)) if not unicodedata.combining(c))


def normalize_name(s: str) -> str:
    import re
    s = _strip_accents(s).lower().strip()
    s = s.replace("°", "")
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^\w]", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

📑 Bloque 5 — Selección del archivo y hoja objetivo

Busca el archivo Excel más reciente dentro de la carpeta indicada, incluso si está bloqueado por OneDrive, creando una copia temporal si es necesario. 

Luego identifica automáticamente la hoja de trabajo adecuada según los nombres preferidos definidos antes.

In [5]:
assert RUTA_ARCHIVOS.is_dir(), f"No existe carpeta: {RUTA_ARCHIVOS}"
archivos = [p for p in RUTA_ARCHIVOS.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
archivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
assert archivos, "No se encontraron Excel válidos."

archivo_origen = archivos[0]
print(f"📄 Archivo más reciente: {archivo_origen.name}  |  {datetime.fromtimestamp(archivo_origen.stat().st_mtime):%Y-%m-%d %H:%M:%S}")

_tmp_copy_path = None

def _norm(s): 
    return " ".join(str(s).split()).lower()

pref_norm = {_norm(n) for n in PREFERRED_SHEETS}

try:
    try:
        with pd.ExcelFile(archivo_origen, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"✅ Hoja objetivo: {target}")

    except PermissionError:
        _tmp_copy_path = archivo_origen.parent / f"__tmp_copy_{int(time.time()*1000)}{archivo_origen.suffix}"
        shutil.copy2(archivo_origen, _tmp_copy_path)
        print(f"📂 No se pudo abrir el archivo original, usando copia temporal: {_tmp_copy_path.name}")

        with pd.ExcelFile(_tmp_copy_path, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"✅ Hoja objetivo (copia): {target}")

except Exception as e:
    raise SystemExit(f"❌ Error al abrir el Excel: {e}")

📄 Archivo más reciente: Base cesantia SURA Stock NOVIEMBRE 2025.xlsx  |  2025-12-29 16:11:56
Hojas: ['Base Stock']
✅ Hoja objetivo: Base Stock


🧼 Bloque 6 — Lectura y limpieza de datos

Lee la hoja seleccionada con pandas y realiza una limpieza básica:
- Convierte todas las columnas a texto (dtype=str).
- Elimina valores vacíos o nulos comunes.
- Normaliza los encabezados de columna a un formato uniforme, sin acentos ni espacios.

In [6]:
def read_excel_safe(io, sheet_name):
    try:
        return pd.read_excel(io, sheet_name=sheet_name, dtype=str, engine="openpyxl")
    except Exception as e:
        raise SystemExit(f"No se pudo leer la hoja '{sheet_name}': {e}")

io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "none": "", "NaN": "", "nan": ""})
df_raw.columns = [normalize_name(c) for c in df_raw.columns]

print("Encabezados normalizados:", list(df_raw.columns))

df_raw.head()

Encabezados normalizados: ['noperacion', 'afinom', 'afirut', 'afirutdv', 'crecuotot', 'cresolmon', 'fecinicob', 'fectercob', 'crecuomon', 'prima_cliente', 'prima_exenta', 'comision_caja_25', 'comision_caja_variable_39', 'producto', 'folio_original', 'fecha_otorgamiento_folio_original']


,noperacion,afinom,afirut,afirutdv,crecuotot,cresolmon,fecinicob,fectercob,crecuomon,prima_cliente,prima_exenta,comision_caja_25,comision_caja_variable_39,producto,folio_original,fecha_otorgamiento_folio_original
0,053000491025,MIGUEL VERA MENA,9933852,0,60,6793662,20231226,20281231,194742,13612,11438.655462184874,2859.6638655462184,4461.075630252101,REPROGRAMACION,053000463217,20140728
1,130000146742,JORGE CABRERA MORALES,12986707,8,36,592643,20221111,20251130,20704,1187,997.4789915966387,249.36974789915968,389.0168067226891,REPROGRAMACION,001027627398,20140916
2,062000149167,ERIK FERNANDEZ FARFAN,11952631,0,60,7635580,20211006,20261031,113136,15299,12856.302521008403,3214.075630252101,5013.957983193278,REPROGRAMACION,022000980054,20140204
3,092001625073,VICTOR CADIN PAILLALAO,16435126,2,22,1677588,20250211,20261231,108599,3361,2824.36974789916,706.09243697479,1101.5042016806724,REPROGRAMACION,003000782661,20140917
4,092001616979,JUAN TRABOL AUCANIR,10846985,4,60,5059584,20230926,20280930,74967,10137,8518.487394957983,2129.621848739496,3322.2100840336134,REPROGRAMACION,102000745843,20140611


🧩 Bloque 7 — Mapeo de columnas y construcción del DataFrame

Relaciona los nombres de las columnas del Excel con los nombres estándar requeridos por la base de datos mediante la función pick().

Luego construye un nuevo DataFrame (df_can) con las columnas limpias y estructuradas, incluyendo una columna adicional con el nombre del archivo original.

In [7]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)

# Origen → Destino
foliocredito        = pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio")
NombreAfiliado      = pick(df_raw, "afinom", "nombre_afiliado", "nombre")
rutafiliado         = pick(df_raw, "afirut", "rut_afiliado", "rut")
dvafiliado          = pick(df_raw, "afirutdv", "dv_afiliado", "dv")
Plazo               = pick(df_raw, "crecuotot", "plazo")
MontoBruto          = pick(df_raw, "cresolmon", "monto_bruto", "monto")
fechaPrimerVto      = pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob")
FechaUltimoVto      = pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob")
ValorCuota          = pick(df_raw, "crecuomon", "valor_cuota")
PrimaBruta_src      = pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_")
PrimaNeta_src       = pick(df_raw, "prima_exenta", "prima_neta")
Com25_src           = pick(df_raw, "comision_caja_25", "comision_caja_25_", "comision_25")
Com26_src           = pick(df_raw, "comision_caja_variable_39", "comision_variable_39", "comision_39")
Producto            = pick(df_raw, "producto")
FolioOrigen         = pick(df_raw, "folio_original", "folio_origen")
FechaOrigen         = pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen")

df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "NombreAfiliado": NombreAfiliado,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "PrimaBruta": PrimaBruta_src,
    "PrimaNeta": PrimaNeta_src,
    "Comision25": Com25_src,
    "ComisionVariable": Com26_src,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "Nombre_de_archivo": archivo_origen.name,
})

df_can.head()

,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
0,053000491025,MIGUEL VERA MENA,9933852,0,60,6793662,20231226,20281231,194742,13612,11438.655462184874,2859.6638655462184,4461.075630252101,REPROGRAMACION,053000463217,20140728,Base cesantia SURA Stock NOVIEMBRE 2025.xlsx
1,130000146742,JORGE CABRERA MORALES,12986707,8,36,592643,20221111,20251130,20704,1187,997.4789915966387,249.36974789915968,389.0168067226891,REPROGRAMACION,001027627398,20140916,Base cesantia SURA Stock NOVIEMBRE 2025.xlsx
2,062000149167,ERIK FERNANDEZ FARFAN,11952631,0,60,7635580,20211006,20261031,113136,15299,12856.302521008403,3214.075630252101,5013.957983193278,REPROGRAMACION,022000980054,20140204,Base cesantia SURA Stock NOVIEMBRE 2025.xlsx
3,092001625073,VICTOR CADIN PAILLALAO,16435126,2,22,1677588,20250211,20261231,108599,3361,2824.36974789916,706.09243697479,1101.5042016806724,REPROGRAMACION,003000782661,20140917,Base cesantia SURA Stock NOVIEMBRE 2025.xlsx
4,092001616979,JUAN TRABOL AUCANIR,10846985,4,60,5059584,20230926,20280930,74967,10137,8518.487394957983,2129.621848739496,3322.2100840336134,REPROGRAMACION,102000745843,20140611,Base cesantia SURA Stock NOVIEMBRE 2025.xlsx


🧾 Bloque 8 — Estandarización del nombre del archivo

Extrae el período YYYYMM desde el nombre del archivo (por ejemplo, "202510" desde “Stock Octubre 2025.xlsx”) y genera un nombre canónico con formato uniforme.

Si el patrón no se encuentra, asigna un nombre alternativo basado en la fecha y hora actual para mantener consistencia.

In [8]:
MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}

MESES_PATTERN = r"(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre|ene|feb|mar|abr|may|jun|jul|ago|sep|set|oct|nov|dic)"

def canonicalizar_nombre_archivo(nombre: str):
    p = Path(nombre)
    stem, ext = p.stem, p.suffix 


    m_per = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem)
    if m_per:
        return stem + ext

    stem_norm = _strip_accents(stem).upper()
    year = None
    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)

    mes_encontrado = None
    for k in MESES.keys():
        if k in stem_norm:
            mes_encontrado = k
            break

    if year and mes_encontrado:
        mm = MESES[mes_encontrado]
        yyyymm = f"{year}{mm}"


        patron = re.compile(rf"{MESES_PATTERN}\s*[-_/.,]*\s*(20\d{{2}})", re.IGNORECASE)
        nuevo_stem, n = patron.subn(yyyymm, stem)
        if n == 0:
            nuevo_stem = f"{stem} {yyyymm}"
        return nuevo_stem + ext
    return stem + ext

nombre_canonico = canonicalizar_nombre_archivo(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)


df_can["Nombre_de_archivo"] = nombre_canonico


Nombre original : Base cesantia SURA Stock NOVIEMBRE 2025.xlsx
Nombre canónico : Base cesantia SURA Stock 202511.xlsx


🧹 Bloque 9 — Eliminación de filas totales

Verifica si la última fila del archivo corresponde a un total general (sumatoria).

Si cumple las condiciones (coincidencia de sumas y alta proporción de celdas vacías), la elimina automáticamente para evitar duplicar datos en la carga final.

In [9]:
def _is_nullish(v):
    if pd.isna(v):
        return True
    if isinstance(v, str):
        s = v.strip()
        return (s == "") or (s.lower() == "none")
    return False


def _nullish_ratio_last_row(df, exclude=()):
    cols = [c for c in df.columns if c not in exclude]
    if not cols:
        return 1.0
    last = df.iloc[-1]
    n_null = sum(_is_nullish(last[c]) for c in cols)
    return n_null / len(cols)


def _round_safe(x, nd):
    try:
        return round(float(x), nd)
    except Exception:
        return np.nan


def drop_last_total_strict(
    df: pd.DataFrame,
    money_cols=("PrimaBruta","PrimaNeta","Comision25","ComisionVariable"),
    round_decimals={"PrimaBruta": 0, "PrimaNeta": 2, "Comision25": 2, "ComisionVariable26": 2},
    null_check_exclude=("Nombre_de_archivo",),
    null_ratio_threshold=0.80,
    atol=0.5,
    rtol=5e-4,
):
    """
    Elimina la última fila si:
      A) Para cada col monetaria: round(last, nd) == round(sum(prev), nd)  O  isclose con (rtol, atol)
      B) En las demás columnas (excepto las excluidas), tiene >= null_ratio_threshold de nulos.
    Muestra la fila eliminada en consola.
    """
    if df is None or df.empty or len(df) < 2:
        print("DF vacío o con <2 filas: nada que eliminar.")
        return df

    last_idx = df.index[-1]
    prev = df.iloc[:-1]
    last = df.iloc[-1]

    # --- A) Chequeo de sumas ---
    sum_flags = {}
    details = {}
    for c in money_cols:
        if c not in df.columns:
            sum_flags[c] = False
            details[c] = "columna no existe"
            continue

        prev_sum = pd.to_numeric(prev[c], errors="coerce").sum(skipna=True)
        last_val = pd.to_numeric(pd.Series([last[c]]), errors="coerce").iloc[0]

        nd = round_decimals.get(c, 2)
        prev_sum_r = _round_safe(prev_sum, nd)
        last_val_r = _round_safe(last_val, nd)

        ok_round = (not np.isnan(prev_sum_r)) and (not np.isnan(last_val_r)) and (last_val_r == prev_sum_r)
        ok_close = (
            np.isfinite(prev_sum) and np.isfinite(last_val) and
            np.isclose(float(last_val), float(prev_sum), rtol=rtol, atol=atol)
        )
        sum_flags[c] = bool(ok_round or ok_close)
        details[c] = f"last={last_val} (r{nd}={last_val_r}) vs sum(prev)={prev_sum} (r{nd}={prev_sum_r})"

    all_sums_ok = all(sum_flags.values())

    # --- B) Chequeo de nulos ---
    null_ratio = _nullish_ratio_last_row(df[[c for c in df.columns if c not in money_cols]], exclude=null_check_exclude)
    nulls_ok = (null_ratio >= null_ratio_threshold)

    # --- C) Decisión ---
    if all_sums_ok and nulls_ok:
        print("🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…\n")
        removed = df.loc[[last_idx]].copy()

        print("🧾 Fila eliminada:")
        display(removed)

        out = df.iloc[:-1].reset_index(drop=True)
        return out

    print("❎ No se eliminó la última fila.")
    print(" - Sumas por columna:")
    for c in money_cols:
        if c in sum_flags:
            print(f"   {c}: match={sum_flags[c]} | {details.get(c)}")
        else:
            print(f"   {c}: columna no existe")
    print(f" - Null ratio última fila (no monetarias): {null_ratio:.2%} (umbral {null_ratio_threshold:.0%})")
    return df

df_can = drop_last_total_strict(df_can)

🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…

🧾 Fila eliminada:


,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
35,,,35,,,,,,70006.88571428572,200472,168463.8655462185,42115.966386554624,65700.90756302522,,,,Base cesantia SURA Stock 202511.xlsx


🧠 Bloque 10 — Tipificación y validación de datos

Convierte cada columna a su tipo de dato correspondiente:
- Números enteros, flotantes o texto según el campo.
- Normaliza la longitud máxima de los textos.
- Verifica valores nulos en columnas críticas.

Finalmente, ordena las columnas para que coincidan con la estructura de la tabla SQL.

In [10]:
INT_COLS    = ["rutafiliado", "Plazo", "fechaPrimerVto", "FechaUltimoVto", "FechaOrigen"]
BIGINT_COLS = ["foliocredito", "FolioOrigen"]
FLOAT_COLS  = ["MontoBruto", "ValorCuota", "PrimaBruta", "PrimaNeta", "Comision25", "ComisionVariable"]
STR_COLS    = ["NombreAfiliado", "Producto", "Nombre_de_archivo"]
DV_COL      = "dvafiliado"


def _to_num_series(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})
    return pd.to_numeric(s2, errors="coerce")

# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")

# DV: char(1) en mayúscula
if DV_COL in df_can.columns:
    df_can[DV_COL] = (
        df_can[DV_COL]
        .astype("string")
        .str.strip()
        .str.upper()
        .map(lambda x: x[:1] if pd.notna(x) and len(x) > 0 else pd.NA)
    )

if "NombreAfiliado" in df_can.columns:
    df_can["NombreAfiliado"] = df_can["NombreAfiliado"].astype("string").str.strip().str.slice(0, 510)

if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 510)

if "Nombre_de_archivo" in df_can.columns:
    df_can["Nombre_de_archivo"] = df_can["Nombre_de_archivo"].astype("string").str.strip().str.slice(0, 50)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

criticas = ["foliocredito","rutafiliado","dvafiliado","FolioOrigen","Nombre_de_archivo"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "foliocredito","NombreAfiliado","rutafiliado","dvafiliado","Plazo","MontoBruto",
    "fechaPrimerVto","FechaUltimoVto","ValorCuota","PrimaBruta","PrimaNeta",
    "Comision25","ComisionVariable","Producto","FolioOrigen","FechaOrigen","Nombre_de_archivo"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

✅ dtypes DESPUÉS:
 foliocredito                  Int64
NombreAfiliado       string[python]
rutafiliado                   Int64
dvafiliado                   object
Plazo                         Int64
MontoBruto                  float64
fechaPrimerVto                Int64
FechaUltimoVto                Int64
ValorCuota                  float64
PrimaBruta                  float64
PrimaNeta                   float64
Comision25                  float64
ComisionVariable            float64
Producto             string[python]
FolioOrigen                   Int64
FechaOrigen                   Int64
Nombre_de_archivo    string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafiliado: 0 nulos
 - dvafiliado: 0 nulos
 - FolioOrigen: 0 nulos
 - Nombre_de_archivo: 0 nulos

📦 df_sql listo con columnas: ['foliocredito', 'NombreAfiliado', 'rutafiliado', 'dvafiliado', 'Plazo', 'MontoBruto', 'fechaPrimerVto', 'FechaUltimoVto', 'ValorCuota', 'PrimaBruta', 'PrimaNeta', 'Com

🔍 Bloque 11 — Validación de unicidad del archivo

Comprueba que el nombre del archivo (Nombre_de_archivo) sea único dentro del DataFrame 

Verifica en la base de datos que aún no exista un registro con ese mismo nombre.

Si ya existe, el proceso se detiene para evitar cargas duplicadas.

In [11]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - Base cesantia SURA Stock 202511.xlsx: 35 filas


🔄 Bloque 12 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [12]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE Nombre_de_archivo = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE Nombre_de_archivo = :nombre
    """)

    for nombre_archivo in vals:
        df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para Nombre_de_archivo = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para Nombre_de_archivo = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para Nombre_de_archivo = 'Base cesantia SURA Stock 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 35 filas para Nombre_de_archivo = 'Base cesantia SURA Stock 202511.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - Base cesantia SURA Stock 202511.xlsx: 35 filas cargadas (archivo nuevo).


🗑️ Bloque 13 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [13]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK STOCK\Base cesantia SURA Stock OCTUBRE 2025.xlsx
